In [1]:
import pandas as pd
import pyarrow

print("pandas:", pd.__version__)
print("pyarrow:", pyarrow.__version__)


pandas: 2.3.3
pyarrow: 20.0.0


In [2]:
file_path = "../data/raw/yellow_tripdata_2025-01.parquet"

df = pd.read_parquet(file_path)

df.head()


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,1,2025-01-01 00:18:38,2025-01-01 00:26:59,1.0,1.60,1.0,N,229,237,1,10.0,3.5,0.5,3.00,0.0,1.0,18.00,2.5,0.0,0.0
1,1,2025-01-01 00:32:40,2025-01-01 00:35:13,1.0,0.50,1.0,N,236,237,1,5.1,3.5,0.5,2.02,0.0,1.0,12.12,2.5,0.0,0.0
2,1,2025-01-01 00:44:04,2025-01-01 00:46:01,1.0,0.60,1.0,N,141,141,1,5.1,3.5,0.5,2.00,0.0,1.0,12.10,2.5,0.0,0.0
3,2,2025-01-01 00:14:27,2025-01-01 00:20:01,3.0,0.52,1.0,N,244,244,2,7.2,1.0,0.5,0.00,0.0,1.0,9.70,0.0,0.0,0.0
4,2,2025-01-01 00:21:34,2025-01-01 00:25:06,3.0,0.66,1.0,N,244,116,2,5.8,1.0,0.5,0.00,0.0,1.0,8.30,0.0,0.0,0.0


In [4]:
df.shape


(3475226, 20)

In [5]:
df.columns


Index(['VendorID', 'tpep_pickup_datetime', 'tpep_dropoff_datetime',
       'passenger_count', 'trip_distance', 'RatecodeID', 'store_and_fwd_flag',
       'PULocationID', 'DOLocationID', 'payment_type', 'fare_amount', 'extra',
       'mta_tax', 'tip_amount', 'tolls_amount', 'improvement_surcharge',
       'total_amount', 'congestion_surcharge', 'Airport_fee',
       'cbd_congestion_fee'],
      dtype='object')

In [6]:
df.dtypes


VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object

# NYC Yellow Taxi Demand Analysis

## 1. Dataset Overview

In [7]:
df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3475226 entries, 0 to 3475225
Data columns (total 20 columns):
 #   Column                 Dtype         
---  ------                 -----         
 0   VendorID               int32         
 1   tpep_pickup_datetime   datetime64[us]
 2   tpep_dropoff_datetime  datetime64[us]
 3   passenger_count        float64       
 4   trip_distance          float64       
 5   RatecodeID             float64       
 6   store_and_fwd_flag     object        
 7   PULocationID           int32         
 8   DOLocationID           int32         
 9   payment_type           int64         
 10  fare_amount            float64       
 11  extra                  float64       
 12  mta_tax                float64       
 13  tip_amount             float64       
 14  tolls_amount           float64       
 15  improvement_surcharge  float64       
 16  total_amount           float64       
 17  congestion_surcharge   float64       
 18  Airport_fee           

## 2. Data Quality Assessment
### 2.1 Missing Values

In [10]:
df.isna().sum()


VendorID                      0
tpep_pickup_datetime          0
tpep_dropoff_datetime         0
passenger_count          540149
trip_distance                 0
RatecodeID               540149
store_and_fwd_flag       540149
PULocationID                  0
DOLocationID                  0
payment_type                  0
fare_amount                   0
extra                         0
mta_tax                       0
tip_amount                    0
tolls_amount                  0
improvement_surcharge         0
total_amount                  0
congestion_surcharge     540149
Airport_fee              540149
cbd_congestion_fee            0
dtype: int64

In [11]:
df.isna().mean() * 100


VendorID                  0.000000
tpep_pickup_datetime      0.000000
tpep_dropoff_datetime     0.000000
passenger_count          15.542845
trip_distance             0.000000
RatecodeID               15.542845
store_and_fwd_flag       15.542845
PULocationID              0.000000
DOLocationID              0.000000
payment_type              0.000000
fare_amount               0.000000
extra                     0.000000
mta_tax                   0.000000
tip_amount                0.000000
tolls_amount              0.000000
improvement_surcharge     0.000000
total_amount              0.000000
congestion_surcharge     15.542845
Airport_fee              15.542845
cbd_congestion_fee        0.000000
dtype: float64

### 2.2 Categorical Variable Validation

In [23]:
df["VendorID"].value_counts(dropna=False)


VendorID
2    2719860
1     753671
7       1206
6        489
Name: count, dtype: int64

In [24]:
df["RatecodeID"].value_counts(dropna=False)


RatecodeID
1.0     2756472
NaN      540149
2.0       94420
99.0      41963
5.0       26501
3.0        8622
4.0        7092
6.0           7
Name: count, dtype: int64

In [25]:
df["payment_type"].value_counts(dropna=False)


payment_type
1    2444393
0     540149
2     390429
4      76481
3      23773
5          1
Name: count, dtype: int64

In [26]:
df["store_and_fwd_flag"].value_counts(dropna=False)


store_and_fwd_flag
N       2927431
None     540149
Y          7646
Name: count, dtype: int64

In [29]:
pd.crosstab(df["payment_type"], df["RatecodeID"], dropna=False)
# it is an over lap
# systematic / structured missingness

RatecodeID,1.0,2.0,3.0,4.0,5.0,6.0,99.0,NaN
payment_type,,,,,,,,
0,0,0,0,0,0,0,0,540149
1,2295290,75440,5873,4890,21088,0,41812,0
2,371098,13481,1598,1468,2671,4,109,0
3,21391,1272,305,113,658,2,32,0
4,68692,4227,846,621,2084,1,10,0
5,1,0,0,0,0,0,0,0


In [30]:
df[df["payment_type"] == 0]["RatecodeID"].isna().mean()


np.float64(1.0)

#### Key Observation: Structured Missingness in RatecodeID

All 540,149 records with `payment_type = 0` have a missing `RatecodeID`, and no other payment type contains missing RatecodeID values.

This suggests that the missingness in `RatecodeID` is systematic rather than random. Therefore, these records should not be removed solely because `RatecodeID` is missing. Further investigation is needed to understand the relationship between Flex Fare records and RatecodeID reporting.

### 2.3 Time Range Validation

Check whether all trip records fall within the expected January 2025 period and identify any abnormal timestamps.

In [31]:
df["tpep_pickup_datetime"].min(), df["tpep_pickup_datetime"].max()


(Timestamp('2024-12-31 20:47:55'), Timestamp('2025-02-01 00:00:44'))

In [32]:
df["tpep_dropoff_datetime"].min(), df["tpep_dropoff_datetime"].max()


(Timestamp('2024-12-18 07:52:40'), Timestamp('2025-02-01 23:44:11'))

In [33]:
(df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]).sum()


np.int64(124)

### 2.3.1 Out-of-Range Timestamp Investigation

The January 2025 file contains timestamps outside the expected monthly window, including records before January 1 and after January 31. We quantify these cases before deciding whether they should be excluded.

In [35]:
jan_start = pd.Timestamp("2025-01-01")
feb_start = pd.Timestamp("2025-02-01")

pickup_outside_jan = (df["tpep_pickup_datetime"] < jan_start) | (
    df["tpep_pickup_datetime"] >= feb_start
)

pickup_outside_jan.sum()


np.int64(22)

In [36]:
df.loc[
    pickup_outside_jan,
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "total_amount"],
].head(20)


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,total_amount
605,2024-12-31 23:30:03,2024-12-31 23:43:02,3.00,26.62
687,2024-12-31 23:31:38,2024-12-31 23:41:48,1.03,15.70
688,2024-12-31 23:46:38,2025-01-01 00:03:03,3.95,24.10
861,2024-12-31 23:56:19,2025-01-01 00:11:19,2.28,23.88
1108,2024-12-31 23:55:37,2025-01-01 00:01:26,1.12,10.40
1312,2024-12-31 23:52:40,2025-01-01 00:23:03,6.72,47.40
2276,2024-12-31 23:49:24,2024-12-31 23:57:30,2.44,20.52
3941,2024-12-31 23:27:13,2024-12-31 23:35:48,1.53,18.00
3942,2024-12-31 23:37:42,2024-12-31 23:43:10,0.92,14.03
4324,2024-12-31 23:51:20,2025-01-01 00:00:00,12.89,69.30


In [37]:
invalid_time_order = df["tpep_dropoff_datetime"] < df["tpep_pickup_datetime"]

df.loc[
    invalid_time_order,
    ["tpep_pickup_datetime", "tpep_dropoff_datetime", "trip_distance", "total_amount"],
].head(20)


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,total_amount
99620,2025-01-02 12:26:00,2025-01-02 11:29:58,9.00,46.00
459885,2025-01-06 16:00:00,2025-01-06 15:05:30,3.80,25.00
1315880,2025-01-15 15:00:00,2025-01-15 14:42:48,1.00,19.00
2029534,2025-01-23 01:44:59,2024-12-18 07:52:40,3.10,30.40
2058555,2025-01-23 12:30:00,2025-01-23 11:44:59,3.90,28.00
2660983,2025-01-29 14:00:00,2025-01-29 13:30:15,1.50,20.00
2945475,2025-01-01 10:01:27,2025-01-01 10:01:00,19.19,68.77
2945514,2025-01-01 10:01:53,2025-01-01 10:01:33,12.17,35.87
2951588,2025-01-02 08:01:41,2025-01-02 08:01:16,14.89,39.11
2952145,2025-01-02 11:01:37,2025-01-02 11:01:29,12.71,34.86


### 2.4 Numerical Variable Validation

Inspect numerical variables for impossible or extreme values before defining cleaning rules.

In [34]:
df[
    [
        "passenger_count",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
        "cbd_congestion_fee",
    ]
].describe().T


,count,mean,std,min,25%,50%,75%,max
passenger_count,2935077.0,1.297859,0.750750,0.00,1.00,1.00,1.00,9.00
trip_distance,3475226.0,5.855126,564.601600,0.00,0.98,1.67,3.10,276423.57
fare_amount,3475226.0,17.081803,463.472918,-900.00,8.60,12.11,19.50,863372.12
tip_amount,3475226.0,2.959813,3.779681,-86.00,0.00,2.45,3.93,400.00
total_amount,3475226.0,25.611292,463.658478,-901.00,15.20,19.95,27.78,863380.37
cbd_congestion_fee,3475226.0,0.483409,0.361931,-0.75,0.00,0.75,0.75,0.75


### 2.4.1 Potential Numerical Anomalies

Extreme and negative values are quantified separately before defining cleaning rules. Negative monetary values may represent reversals or adjustments, while implausibly large trip distances or fares are more likely to indicate erroneous records.

In [38]:
checks = {
    "zero_or_negative_distance": (df["trip_distance"] <= 0).sum(),
    "distance_over_100_miles": (df["trip_distance"] > 100).sum(),
    "negative_fare": (df["fare_amount"] < 0).sum(),
    "fare_over_500": (df["fare_amount"] > 500).sum(),
    "negative_total": (df["total_amount"] < 0).sum(),
    "total_over_500": (df["total_amount"] > 500).sum(),
    "zero_passenger": (df["passenger_count"] == 0).sum(),
}

pd.Series(checks)


zero_or_negative_distance     90893
distance_over_100_miles         162
negative_fare                144118
fare_over_500                    55
negative_total                63037
total_over_500                   79
zero_passenger                24656
dtype: int64

In [39]:
df.nlargest(10, "trip_distance")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "PULocationID",
        "DOLocationID",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ]
]


,tpep_pickup_datetime,tpep_dropoff_datetime,PULocationID,DOLocationID,trip_distance,fare_amount,total_amount
3275762,2025-01-22 21:04:00,2025-01-22 21:13:00,161,170,276423.57,-4.75,5.00
3188501,2025-01-19 14:51:00,2025-01-19 14:58:00,224,233,276099.95,9.13,13.88
3240337,2025-01-21 09:19:00,2025-01-21 09:45:00,52,144,222167.49,31.19,35.94
3112173,2025-01-16 10:47:00,2025-01-16 11:11:00,263,48,206137.99,24.89,29.64
3374854,2025-01-26 17:46:00,2025-01-26 17:59:00,50,186,202771.63,10.10,14.85
3081597,2025-01-14 19:18:00,2025-01-14 19:36:00,246,164,189687.43,12.70,17.45
3069039,2025-01-13 21:11:00,2025-01-13 21:17:00,100,90,181139.99,6.33,11.08
3312563,2025-01-24 08:33:00,2025-01-24 08:44:00,42,238,168079.57,-4.00,4.49
3195037,2025-01-19 18:31:00,2025-01-19 18:58:00,247,239,167452.94,-4.00,5.45
3138510,2025-01-17 14:25:00,2025-01-17 14:40:00,262,239,164959.95,14.05,18.05


In [40]:
df.nlargest(10, "total_amount")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_distance",
        "fare_amount",
        "tip_amount",
        "total_amount",
    ]
]


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_distance,fare_amount,tip_amount,total_amount
1780915,2025-01-20 12:07:18,2025-01-20 12:12:42,1.60,863372.12,0.0,863380.37
1404958,2025-01-16 12:23:14,2025-01-17 14:41:56,255.33,2450.90,0.0,2506.71
1832645,2025-01-21 05:17:57,2025-01-21 08:19:45,188.88,1309.20,0.0,1311.70
1346041,2025-01-15 19:14:42,2025-01-15 21:46:13,143.54,936.80,0.0,969.05
2358832,2025-01-26 00:04:04,2025-01-26 00:04:42,0.00,950.00,0.0,953.50
2358836,2025-01-26 00:22:47,2025-01-26 00:22:55,0.00,950.00,0.0,951.00
2358837,2025-01-26 00:24:25,2025-01-26 00:24:35,0.00,950.00,0.0,951.00
2358828,2025-01-26 00:00:56,2025-01-26 00:01:06,0.00,900.00,0.0,903.50
2353880,2025-01-25 23:56:58,2025-01-25 23:57:06,0.00,899.99,0.0,903.49
2353881,2025-01-25 23:58:50,2025-01-25 23:59:00,0.00,899.99,0.0,903.49


In [42]:
df["trip_duration_min"] = (
    df["tpep_dropoff_datetime"] - df["tpep_pickup_datetime"]
).dt.total_seconds() / 60


In [43]:
df["trip_duration_min"].describe()


count    3.475226e+06
mean     1.501812e+01
std      3.871358e+01
min     -5.147232e+04
25%      7.283333e+00
50%      1.170000e+01
75%      1.833333e+01
max      5.626317e+03
Name: trip_duration_min, dtype: float64

In [44]:
(df["trip_duration_min"] <= 0).sum()


np.int64(2051)

## 2.5 Data Quality Flags

Instead of immediately deleting suspicious records, quality flags are created to distinguish clearly invalid observations from records that may reflect legitimate operational behavior.

These flags will be used to define different analytical samples for demand, fare, and modeling tasks.

In [45]:
df["flag_outside_jan"] = (df["tpep_pickup_datetime"] < "2025-01-01") | (
    df["tpep_pickup_datetime"] >= "2025-02-01"
)

df["flag_invalid_duration"] = df["trip_duration_min"] <= 0

df["flag_extreme_distance"] = df["trip_distance"] > 100

df["flag_nonpositive_distance"] = df["trip_distance"] <= 0

df["flag_negative_fare"] = df["fare_amount"] < 0

df["flag_extreme_fare"] = df["fare_amount"] > 500


In [46]:
quality_summary = df[
    [
        "flag_outside_jan",
        "flag_invalid_duration",
        "flag_extreme_distance",
        "flag_nonpositive_distance",
        "flag_negative_fare",
        "flag_extreme_fare",
    ]
].sum()

quality_summary


flag_outside_jan                 22
flag_invalid_duration          2051
flag_extreme_distance           162
flag_nonpositive_distance     90893
flag_negative_fare           144118
flag_extreme_fare                55
dtype: int64

In [47]:
duration_checks = {
    "over_2_hours": (df["trip_duration_min"] > 120).sum(),
    "over_3_hours": (df["trip_duration_min"] > 180).sum(),
    "over_4_hours": (df["trip_duration_min"] > 240).sum(),
    "over_6_hours": (df["trip_duration_min"] > 360).sum(),
}

pd.Series(duration_checks)


over_2_hours    2014
over_3_hours    1377
over_4_hours    1278
over_6_hours    1203
dtype: int64

In [48]:
df.nlargest(20, "trip_duration_min")[
    [
        "tpep_pickup_datetime",
        "tpep_dropoff_datetime",
        "trip_duration_min",
        "trip_distance",
        "fare_amount",
        "total_amount",
    ]
]


,tpep_pickup_datetime,tpep_dropoff_datetime,trip_duration_min,trip_distance,fare_amount,total_amount
1858816,2025-01-21 12:23:41,2025-01-25 10:10:00,5626.316667,6.55,-28.20,-36.45
1858817,2025-01-21 12:23:41,2025-01-25 10:10:00,5626.316667,6.55,28.20,36.45
414374,2025-01-06 02:12:11,2025-01-09 13:54:19,5022.133333,91.51,336.90,339.40
2460536,2025-01-27 11:58:55,2025-01-30 09:42:55,4184.000000,0.00,3.00,4.50
898201,2025-01-11 08:04:00,2025-01-13 16:49:28,3405.466667,0.04,70.00,76.50
2147536,2025-01-24 07:07:21,2025-01-26 10:09:44,3062.383333,0.00,3.00,4.50
85681,2025-01-02 09:40:31,2025-01-04 09:37:01,2876.500000,0.00,3.00,4.50
1536147,2025-01-17 16:45:49,2025-01-19 16:40:59,2875.166667,1.98,12.80,20.05
556772,2025-01-07 18:07:33,2025-01-09 16:12:05,2764.533333,1.86,47.80,51.80
3417048,2025-01-29 21:39:23,2025-01-31 16:48:43,2589.333333,7.52,31.02,35.77


### 2.5.1 Initial Data Quality Findings

Several data-quality patterns were identified:

- A small number of records fall outside the expected January 2025 pickup window.
- 2,051 trips have non-positive trip durations.
- Extremely large trip distances and fares indicate clear data-entry or reporting errors.
- Some negative-fare records appear in positive/negative pairs with identical trip characteristics, suggesting possible reversals or transaction corrections rather than simple invalid records.
- Very long trip durations may reflect meter or timestamp issues and should be treated separately from normal demand records.

These observations motivate the use of task-specific cleaning rules rather than applying a single global filter to the entire dataset.

## location validation

In [49]:
zones = pd.read_csv("../data/raw/taxi_zone_lookup.csv")


In [50]:
zones.head()


,LocationID,Borough,Zone,service_zone
0,1,EWR,Newark Airport,EWR
1,2,Queens,Jamaica Bay,Boro Zone
2,3,Bronx,Allerton/Pelham Gardens,Boro Zone
3,4,Manhattan,Alphabet City,Yellow Zone
4,5,Staten Island,Arden Heights,Boro Zone


In [51]:
zones.shape


(265, 4)

In [52]:
zones["LocationID"].nunique()


265

In [53]:
df["PULocationID"]


0          229
1          236
2          141
3          244
4          244
          ... 
3475221     79
3475222    161
3475223    144
3475224    142
3475225    237
Name: PULocationID, Length: 3475226, dtype: int32

In [54]:
zones["LocationID"]


0        1
1        2
2        3
3        4
4        5
      ... 
260    261
261    262
262    263
263    264
264    265
Name: LocationID, Length: 265, dtype: int64

In [55]:
df["PULocationID"].isin(zones["LocationID"]).value_counts()


PULocationID
True    3475226
Name: count, dtype: int64

In [56]:
df["DOLocationID"].isin(zones["LocationID"]).value_counts()


DOLocationID
True    3475226
Name: count, dtype: int64

In [57]:
df.loc[~df["PULocationID"].isin(zones["LocationID"]), "PULocationID"].value_counts()


Series([], Name: count, dtype: int64)